# **DATASET LOADING AND CSV CREATION**

In [ ]:
pip install datasets


In [ ]:
from datasets import load_dataset

# Load your dataset
ds = load_dataset("Bhargav1127/necklace-dataset-with-captions")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/310 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/329k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_dataset
import pandas as pd
import os
from tqdm import tqdm

# 2. Make a folder for images
save_dir = "necklace_images"
os.makedirs(save_dir, exist_ok=True)

# 3. Save images + build CSV rows
rows = []
for i, example in enumerate(tqdm(ds["train"])):
    img = example["image"]       # already a PIL Image
    caption = example["caption"]

    # save image as JPG
    img_path = f"{save_dir}/{i}.jpg"
    img.save(img_path)

    rows.append({"image": img_path, "caption": caption})

# 4. Save CSV
df = pd.DataFrame(rows)
df.to_csv("necklace_dataset.csv", index=False)

print("✅ Done! Images saved in 'necklace_images/' and CSV saved as 'necklace_dataset.csv'")


100%|██████████| 100/100 [00:00<00:00, 531.59it/s]

✅ Done! Images saved in 'necklace_images/' and CSV saved as 'necklace_dataset.csv'


In [ ]:
from datasets import load_dataset

# Load your dataset
ds = load_dataset("imnaveenk/necklace")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/388 [00:00<?, ?B/s]

(…)-00000-of-00001-bb50e3f9df1f5428.parquet:   0%|          | 0.00/36.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21 [00:00<?, ? examples/s]

In [ ]:
import shutil
from google.colab import files

# 1. Zip the folder
shutil.make_archive("necklace_images", 'zip', "necklace_images")

# 2. Download the zip file
files.download("necklace_images.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **COMBINING THE 3 DATASETS AND FILTERING FOR NECKLACE AND EARRINGS**

In [ ]:
import pandas as pd

# ---------- STEP 1: Safe Loader ----------
def load_csv(path):
    """Load CSV safely with encoding fallback"""
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin1")

# ---------- STEP 2: Load All Datasets ----------
dataset1 = load_csv("/content/dataset_labels.csv")       # CSV
dataset2 = pd.read_excel("/content/naveen_dataset.xlsx") # XLSX
dataset3 = pd.read_excel("/content/necklace_dataset.xlsx") # XLSX

print("Dataset 1 columns:", dataset1.columns.tolist())
print("Dataset 2 columns:", dataset2.columns.tolist())
print("Dataset 3 columns:", dataset3.columns.tolist())

# ---------- STEP 3: Define Jewelry Filters ----------
# Whitelist: what we want
keep_keywords = ["necklace", "necklaces", "earring", "earrings", "stud", "studs", "hoop", "jhumka"]

# Blacklist: what we want to exclude
exclude_keywords = ["bracelet", "bracelets", "ring", "rings", "bangle", "bangles", "anklet", "anklets", "pendant", "pendants"]

def filter_jewelry(df, text_col, image_col, dataset_name):
    """
    Filters rows to include ONLY necklace/earring and exclude other jewelry types.
    """
    df[text_col] = df[text_col].astype(str).str.lower()

    # Must contain a keep keyword
    keep_mask = df[text_col].str.contains("|".join(keep_keywords), case=False, na=False)

    # Must NOT contain an exclude keyword
    exclude_mask = df[text_col].str.contains("|".join(exclude_keywords), case=False, na=False)

    mask = keep_mask & ~exclude_mask

    filtered = df.loc[mask, [text_col, image_col]].copy()
    filtered["source_dataset"] = dataset_name
    return filtered

# ---------- STEP 4: Apply Filter ----------
# Replace column names based on your dataset
filtered1 = filter_jewelry(dataset1, text_col="description", image_col="image_path", dataset_name="dataset1")
filtered2 = filter_jewelry(dataset2, text_col="caption", image_col="image", dataset_name="dataset2")
filtered3 = filter_jewelry(dataset3, text_col="caption", image_col="image", dataset_name="dataset3")

# ---------- STEP 5: Combine & Save ----------
final_df = pd.concat([filtered1, filtered2, filtered3], ignore_index=True)

# Rename consistently
final_df.rename(columns={
    "caption": "text",
    "description": "text",
    "image_path": "image"
}, inplace=True)

# Save
final_df.to_csv("final_necklace_earring_dataset.csv", index=False)

print("✅ Final dataset saved as final_necklace_earring_dataset.csv")
print("Total rows:", len(final_df))
final_df.head()

Dataset 1 columns: ['image_path', 'description']
Dataset 2 columns: ['image', 'caption']
Dataset 3 columns: ['image', 'caption']
✅ Final dataset saved as final_necklace_earring_dataset.csv
Total rows: 613


,text,image,source_dataset,text,image
0,"a single, large pearl centered on a black cord...",bracelet\365-00492_1400x.jpg,dataset1,NaN,NaN
1,a delicate silver chain necklace with a twiste...,bracelet\9CIWHEQYX04B.jpg,dataset1,NaN,NaN
2,a delicate gold necklace adorned with lustrous...,bracelet\bracelet_123.jpg,dataset1,NaN,NaN
3,a delicate pearl necklace with a gold clasp.,bracelet\FO9PYVZIRUHT.jpg,dataset1,NaN,NaN
4,five strands of pearls in a gold choker neckla...,bracelet\HGBJ9HJZEOZN.jpg,dataset1,NaN,NaN


In [ ]:
import pandas as pd
import re
from pathlib import Path

def load_csv_with_encoding(filename):
    """Try to load CSV with different encodings"""
    encodings = ['utf-8', 'latin1', 'iso-8859-1', 'cp1252', 'utf-16']

    for encoding in encodings:
        try:
            print(f"Trying encoding: {encoding}")
            df = pd.read_csv(filename, encoding=encoding)
            print(f"Successfully loaded with {encoding} encoding")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"Error with {encoding}: {e}")
            continue

    # If all encodings fail, try with error handling
    try:
        print("Trying with error handling (replacing bad characters)...")
        df = pd.read_csv(filename, encoding='utf-8', errors='replace')
        print("Loaded with character replacement")
        return df
    except Exception as e:
        print(f"Final attempt failed: {e}")
        return None

def load_datasets():
    """Load the three datasets with proper handling for different file types"""
    datasets = []

    # Dataset 1 - CSV (needs filtering)
    print("Loading Dataset 1 (CSV)...")
    df1 = load_csv_with_encoding('/content/dataset_labels.csv')  # Replace with your actual filename
    if df1 is not None:
        print(f"Dataset 1 loaded: {len(df1)} rows")
        datasets.append(('CSV', df1, True))  # True means needs filtering
    else:
        print("Failed to load CSV file")

    # Dataset 2 - XLSX (already contains only necklaces)
    print("Loading Dataset 2 (XLSX)...")
    try:
        df2 = pd.read_excel('/content/naveen_dataset.xlsx')  # Replace with your actual filename
        print(f"Dataset 2 loaded: {len(df2)} rows")
        datasets.append(('XLSX1', df2, False))  # False means no filtering needed
    except Exception as e:
        print(f"Error loading first XLSX: {e}")

    # Dataset 3 - XLSX (already contains only necklaces)
    print("Loading Dataset 3 (XLSX)...")
    try:
        df3 = pd.read_excel('/content/necklace_dataset.xlsx')  # Replace with your actual filename
        print(f"Dataset 3 loaded: {len(df3)} rows")
        datasets.append(('XLSX2', df3, False))  # False means no filtering needed
    except Exception as e:
        print(f"Error loading second XLSX: {e}")

    return datasets

def identify_columns(df, dataset_name):
    """Identify image path and description columns"""
    columns = df.columns.tolist()
    print(f"\n{dataset_name} columns: {columns}")

    # Look for image path column
    image_col = None
    desc_col = None

    for col in columns:
        col_lower = col.lower().strip()
        # Check for image path column
        if any(keyword in col_lower for keyword in ['image_path', 'image', 'path', 'filename']):
            if image_col is None:  # Take the first match
                image_col = col

        # Check for description column
        if any(keyword in col_lower for keyword in ['description', 'caption', 'desc']):
            if desc_col is None:  # Take the first match
                desc_col = col

    print(f"Identified - Image column: {image_col}, Description column: {desc_col}")
    return image_col, desc_col

def is_necklace_or_earring_from_path_only(image_path):
    """Check if the image path indicates necklace or earring (CSV dataset only - path only)"""
    if pd.isna(image_path):
        return False

    image_path_str = str(image_path).lower()

    # Look for folder names or file names that indicate jewelry type
    target_keywords = ['necklace', 'necklaces', 'earring', 'earrings', 'earring_best']

    return any(keyword in image_path_str for keyword in target_keywords)

def get_jewelry_type_from_path_only(image_path):
    """Determine jewelry type from image path only (for CSV dataset)"""
    if pd.isna(image_path):
        return 'unknown'

    image_path_str = str(image_path).lower()

    # Check for earring keywords first (more specific)
    earring_keywords = ['earring', 'earrings', 'earring_best']
    if any(keyword in image_path_str for keyword in earring_keywords):
        return 'earring'

    # Check for necklace keywords
    necklace_keywords = ['necklace', 'necklaces']
    if any(keyword in image_path_str for keyword in necklace_keywords):
        return 'necklace'

    return 'unknown'

def process_datasets():
    """Main processing function"""
    datasets = load_datasets()

    if not datasets:
        print("No datasets loaded successfully!")
        return None

    combined_data = []

    for dataset_name, df, needs_filtering in datasets:
        print(f"\nProcessing {dataset_name}...")

        # Identify the relevant columns
        image_col, desc_col = identify_columns(df, dataset_name)

        if not image_col or not desc_col:
            print(f"Could not identify required columns in {dataset_name}")
            continue

        if needs_filtering:
            # This is the CSV dataset - filter based on image path only
            print("Filtering CSV dataset based on image path only...")
            def filter_csv_jewelry(row):
                return is_necklace_or_earring_from_path_only(row[image_col])

            filtered_df = df[df.apply(filter_csv_jewelry, axis=1)].copy()
            print(f"Found {len(filtered_df)} necklaces/earrings out of {len(df)} total items")

            if len(filtered_df) > 0:
                # Add jewelry type column based on path only
                def classify_csv_jewelry(row):
                    return get_jewelry_type_from_path_only(row[image_col])

                filtered_df['jewelry_type'] = filtered_df.apply(classify_csv_jewelry, axis=1)
        else:
            # This is an XLSX dataset - take as is (already contains only necklaces)
            print("Taking XLSX dataset as is (already contains only necklaces)...")
            filtered_df = df.copy()

            # Since XLSX files already contain only necklaces, set jewelry_type to 'necklace'
            filtered_df['jewelry_type'] = 'necklace'

        if len(filtered_df) > 0:
            # Rename columns to standard names
            filtered_df = filtered_df.rename(columns={
                image_col: 'image_path',
                desc_col: 'description'
            })

            # Add source dataset column
            filtered_df['source_dataset'] = dataset_name

            # Select only needed columns
            final_cols = ['image_path', 'description', 'jewelry_type', 'source_dataset']
            filtered_df = filtered_df[final_cols]

            combined_data.append(filtered_df)
            print(f"Added {len(filtered_df)} items from {dataset_name}")
        else:
            print(f"No valid items found in {dataset_name}")

    if not combined_data:
        print("No data to combine!")
        return None

    # Combine all datasets
    final_df = pd.concat(combined_data, ignore_index=True)

    print(f"\nFinal combined dataset:")
    print(f"Total items: {len(final_df)}")
    print(f"Necklaces: {len(final_df[final_df['jewelry_type'] == 'necklace'])}")
    print(f"Earrings: {len(final_df[final_df['jewelry_type'] == 'earring'])}")
    print(f"Unknown: {len(final_df[final_df['jewelry_type'] == 'unknown'])}")

    # Show sample data
    print(f"\nSample data:")
    print(final_df.head())

    # Show distribution by source
    print(f"\nDistribution by source:")
    print(final_df.groupby(['source_dataset', 'jewelry_type']).size().unstack(fill_value=0))

    return final_df

def save_results(df, filename='combined_jewelry_dataset.csv'):
    """Save the final dataset"""
    if df is not None:
        df.to_csv(filename, index=False)
        print(f"\nDataset saved as '{filename}'")
        return True
    return False

# Main execution
if __name__ == "__main__":
    print("Starting jewelry dataset processing...")
    print("Make sure your dataset files are uploaded to Colab!")
    print("\nUpdate the filenames in the load_datasets() function:")
    print("- dataset_labels.csv (replace with your CSV filename)")
    print("- naveen_dataset.xlsx (replace with your first XLSX filename)")
    print("- necklace_dataset.xlsx (replace with your second XLSX filename)")

    # Process the datasets
    final_dataset = process_datasets()

    # Save results
    if final_dataset is not None:
        save_results(final_dataset)

        # Additional analysis
        print("\n" + "="*50)
        print("SUMMARY STATISTICS")
        print("="*50)
        print("Jewelry type distribution:")
        print(final_dataset['jewelry_type'].value_counts())
        print(f"\nDataset distribution:")
        print(final_dataset['source_dataset'].value_counts())

        # Show some examples of filtered items from CSV
        csv_items = final_dataset[final_dataset['source_dataset'] == 'CSV']
        if len(csv_items) > 0:
            print(f"\nSample CSV items (filtered by image path):")
            print(csv_items[['image_path', 'jewelry_type']].head())
    else:
        print("Processing failed - no output generated")

# Quick test function to check your data structure
def preview_datasets():
    """Quick preview of your datasets to check structure"""
    datasets = load_datasets()
    for name, df, needs_filtering in datasets:
        print(f"\n{name} Preview (Needs filtering: {needs_filtering}):")
        print("Columns:", df.columns.tolist())
        print("Shape:", df.shape)
        print("First few rows:")
        print(df.head(2))
        print("-" * 40)

# Uncomment the line below to preview your datasets first
# preview_datasets()

Starting jewelry dataset processing...
Make sure your dataset files are uploaded to Colab!

Update the filenames in the load_datasets() function:
- dataset_labels.csv (replace with your CSV filename)
- naveen_dataset.xlsx (replace with your first XLSX filename)
- necklace_dataset.xlsx (replace with your second XLSX filename)
Loading Dataset 1 (CSV)...
Trying encoding: utf-8
Trying encoding: latin1
Successfully loaded with latin1 encoding
Dataset 1 loaded: 6156 rows
Loading Dataset 2 (XLSX)...
Dataset 2 loaded: 21 rows
Loading Dataset 3 (XLSX)...
Dataset 3 loaded: 100 rows

Processing CSV...

CSV columns: ['image_path', 'description']
Identified - Image column: image_path, Description column: description
Filtering CSV dataset based on image path only...
Found 5035 necklaces/earrings out of 6156 total items
Added 5035 items from CSV

Processing XLSX1...

XLSX1 columns: ['image', 'caption']
Identified - Image column: image, Description column: caption
Taking XLSX dataset as is (already co

# **Augmentation**

In [ ]:
!pip install -q tensorflow

In [ ]:
# Jewelry Dataset Augmentation for Google Colab
# Augments images based on CSV file with image_path, description, type columns

import pandas as pd
import numpy as np
import os
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, save_img
import matplotlib.pyplot as plt
from PIL import Image
import shutil
from tqdm import tqdm
import random
import chardet  # For encoding detection

# Mount Google Drive (uncomment if your data is in Google Drive)
# from google.colab import drive
# drive.mount('/content/drive')

# Configuration
CSV_FILE_PATH = '/content/your_dataset.csv'  # Update this path
BASE_OUTPUT_DIR = '/content/augmented_dataset'  # Where to save augmented images

# Multiple image directories - Update these with your folder paths
IMAGE_FOLDERS = [
    '/content/folder1',  # Update with your folder names
    '/content/folder2',
    '/content/folder3',
    '/content/folder4'
]

TARGET_SIZE = (224, 224)  # Resize images to this size
AUGMENTATIONS_PER_IMAGE = 8  # Number of augmented versions per original image

# Create output directories
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{BASE_OUTPUT_DIR}/images', exist_ok=True)

# Function to detect and load CSV with proper encoding
def load_csv_with_encoding(file_path):
    """Load CSV file with automatic encoding detection"""
    import chardet

    # Detect encoding
    with open(file_path, 'rb') as f:
        raw_data = f.read(10000)  # Read first 10KB to detect encoding
        encoding_result = chardet.detect(raw_data)
        detected_encoding = encoding_result['encoding']
        confidence = encoding_result['confidence']

    print(f"Detected encoding: {detected_encoding} (confidence: {confidence:.2f})")

    # Try different encodings in order of preference
    encodings_to_try = [detected_encoding, 'utf-8', 'latin-1', 'cp1252', 'iso-8859-1']

    for encoding in encodings_to_try:
        if encoding is None:
            continue
        try:
            print(f"Trying to load with {encoding} encoding...")
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"Successfully loaded with {encoding} encoding!")
            return df
        except UnicodeDecodeError as e:
            print(f"Failed with {encoding}: {e}")
            continue
        except Exception as e:
            print(f"Error with {encoding}: {e}")
            continue

    # If all encodings fail, try with error handling
    try:
        print("Trying with error handling (replacing bad characters)...")
        df = pd.read_csv(file_path, encoding='utf-8', encoding_errors='replace')
        print("Successfully loaded with error handling!")
        return df
    except Exception as e:
        raise Exception(f"Could not load CSV file with any encoding method: {e}")

# Load the CSV file
print("Loading CSV file...")
df = load_csv_with_encoding(CSV_FILE_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Sample data:")
print(df.head())

# Define augmentation parameters based on your requirements
datagen = ImageDataGenerator(
    rotation_range=90,           # 90° rotations
    width_shift_range=0.3,       # Image width shift by 30%
    height_shift_range=0.3,      # Image length shift by 30%
    zoom_range=[0.85, 1.05],     # Image enlargement/reduction by 15% (actually using zoom for "cuts")
    brightness_range=[0.2, 1.8], # Brightness range by 80% (0.2 = 80% darker, 1.8 = 80% brighter)
    horizontal_flip=True,        # Horizontal flips
    vertical_flip=True,          # Vertical flips
    fill_mode='nearest'          # How to fill in newly created pixels
)

# Additional augmentation function for color changes
def apply_color_shift(image_array, intensity=0.1):
    """Apply slight color changes to the image"""
    # Convert to float for processing
    img = image_array.astype(np.float32) / 255.0

    # Random color shifts for each channel
    color_shifts = np.random.uniform(-intensity, intensity, 3)

    for i in range(3):  # RGB channels
        img[:, :, i] = np.clip(img[:, :, i] + color_shifts[i], 0, 1)

    return (img * 255).astype(np.uint8)

# Function to find image in multiple folders
def find_image_path(image_filename, search_folders):
    """Find the full path of an image across multiple folders"""
    # If it's already a full path, return it
    if os.path.isabs(image_filename) and os.path.exists(image_filename):
        return image_filename

    # Extract just the filename if path includes folders
    base_filename = os.path.basename(image_filename)

    # Search in all folders
    for folder in search_folders:
        potential_path = os.path.join(folder, base_filename)
        if os.path.exists(potential_path):
            return potential_path

        # Also try the original filename in case it includes subfolder
        potential_path = os.path.join(folder, image_filename)
        if os.path.exists(potential_path):
            return potential_path

    return None
    """Create a new filename for augmented image"""
    base_name = os.path.splitext(os.path.basename(original_path))[0]
    extension = os.path.splitext(original_path)[1]
    return f"{base_name}_aug_{aug_index}_{aug_type}{extension}"

def process_single_image(img_path, output_dir, base_filename):
    """Process a single image and create augmented versions"""
    try:
        # Load and preprocess image
        if not os.path.exists(img_path):
            print(f"Warning: Image not found - {img_path}")
            return []

        # Load image
        img = load_img(img_path, target_size=TARGET_SIZE)
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)

        augmented_files = []

        # Generate augmented images
        aug_iter = datagen.flow(img_array, batch_size=1)

        for i in range(AUGMENTATIONS_PER_IMAGE):
            # Get augmented image
            aug_img = next(aug_iter)[0].astype(np.uint8)

            # Apply additional color shift (50% chance)
            if random.random() > 0.5:
                aug_img = apply_color_shift(aug_img, intensity=0.1)
                aug_type = f"color_shift_{i}"
            else:
                aug_type = f"geometric_{i}"

            # Create filename and save
            aug_filename = create_augmented_filename(base_filename, i, aug_type)
            aug_path = os.path.join(output_dir, aug_filename)

            # Save augmented image
            save_img(aug_path, aug_img)
            augmented_files.append(aug_filename)

        return augmented_files

    except Exception as e:
        print(f"Error processing {img_path}: {str(e)}")
        return []

# Create augmented dataset
print("Starting augmentation process...")
augmented_data = []
missing_images = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Augmenting images"):
    original_img_path = row['image_path']

    # Find the actual image path across multiple folders
    full_img_path = find_image_path(original_img_path, IMAGE_FOLDERS)

    if full_img_path is None:
        print(f"Warning: Image not found in any folder - {original_img_path}")
        missing_images.append(original_img_path)
        continue

    # Process the image
    base_filename = os.path.basename(original_img_path)
    augmented_files = process_single_image(full_img_path, f'{BASE_OUTPUT_DIR}/images', base_filename)

    # Add original image to new dataset
    augmented_data.append({
        'image_path': f'images/{base_filename}',
        'description': row['description'],
        'type': row['type'],
        'is_augmented': False,
        'original_image': base_filename,
        'source_folder': os.path.dirname(full_img_path)
    })

    # Copy original image to output directory
    try:
        shutil.copy2(full_img_path, f'{BASE_OUTPUT_DIR}/images/{base_filename}')
    except Exception as e:
        print(f"Error copying original image {full_img_path}: {str(e)}")

    # Add augmented images to dataset
    for aug_file in augmented_files:
        augmented_data.append({
            'image_path': f'images/{aug_file}',
            'description': row['description'],  # Keep original description
            'type': row['type'],  # Keep original type
            'is_augmented': True,
            'original_image': base_filename,
            'source_folder': os.path.dirname(full_img_path)
        })

# Create new DataFrame with augmented data
augmented_df = pd.DataFrame(augmented_data)

# Save the new CSV
output_csv_path = f'{BASE_OUTPUT_DIR}/augmented_dataset.csv'
augmented_df.to_csv(output_csv_path, index=False)

print(f"\nAugmentation completed!")
print(f"Original dataset size: {len(df)} images")
print(f"Augmented dataset size: {len(augmented_df)} images")
print(f"Missing images: {len(missing_images)}")
if missing_images:
    print("Missing image files:")
    for missing in missing_images[:10]:  # Show first 10
        print(f"  - {missing}")
    if len(missing_images) > 10:
        print(f"  ... and {len(missing_images) - 10} more")

print(f"Augmentation factor: {len(augmented_df) / (len(df) - len(missing_images)):.2f}x")
print(f"\nFiles saved to: {BASE_OUTPUT_DIR}")
print(f"CSV file: {output_csv_path}")

# Display statistics
print(f"\nDataset statistics:")
print(f"Original images: {len(augmented_df[augmented_df['is_augmented'] == False])}")
print(f"Augmented images: {len(augmented_df[augmented_df['is_augmented'] == True])}")
print(f"\nType distribution:")
print(augmented_df['type'].value_counts())
print(f"\nSource folder distribution:")
print(augmented_df[augmented_df['is_augmented'] == False]['source_folder'].value_counts())

# Function to visualize some augmentations (optional)
def visualize_augmentations(original_img_path, num_augmentations=4):
    """Visualize original image and its augmentations"""
    if not os.path.exists(original_img_path):
        print(f"Image not found: {original_img_path}")
        return

    # Load original image
    orig_img = load_img(original_img_path, target_size=TARGET_SIZE)
    orig_array = img_to_array(orig_img)
    orig_array = np.expand_dims(orig_array, axis=0)

    # Create figure
    fig, axes = plt.subplots(1, num_augmentations + 1, figsize=(20, 4))

    # Show original
    axes[0].imshow(orig_img)
    axes[0].set_title('Original')
    axes[0].axis('off')

    # Generate and show augmentations
    aug_iter = datagen.flow(orig_array, batch_size=1)

    for i in range(num_augmentations):
        aug_img = next(aug_iter)[0].astype(np.uint8)
        axes[i + 1].imshow(aug_img)
        axes[i + 1].set_title(f'Augmented {i + 1}')
        axes[i + 1].axis('off')

    plt.tight_layout()
    plt.show()

# Example usage of visualization (uncomment to use)
# if len(augmented_df) > 0:
#     # Find a sample image that exists
#     sample_row = augmented_df[augmented_df['is_augmented'] == False].iloc[0]
#     sample_img_path = find_image_path(sample_row['original_image'], IMAGE_FOLDERS)
#     if sample_img_path:
#         print(f"Visualizing augmentations for: {sample_img_path}")
#         visualize_augmentations(sample_img_path)

print("\nScript completed successfully!")
print(f"You can now use the augmented dataset from: {output_csv_path}")

In [ ]:
# Jewelry Dataset Augmentation for Google Colab
# Augments images based on CSV file with image_path, description, type columns

import pandas as pd
import numpy as np
import os
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, save_img
import matplotlib.pyplot as plt
from PIL import Image
import shutil
from tqdm import tqdm
import random

# Mount Google Drive (uncomment if your data is in Google Drive)
# from google.colab import drive
# drive.mount('/content/drive')

# Configuration
CSV_FILE_PATH = '/content/combined_jewelry_dataset - combined_jewelry_dataset.csv'  # Update this path
BASE_OUTPUT_DIR = '/content/Augmentation'  # Where to save augmented images

# Multiple image directories - Update these with your folder paths
IMAGE_FOLDERS = [
    '/content/drive/MyDrive/dataset/necklace_images',  # Update with your folder names
    '/content/drive/MyDrive/dataset/necklace',
    '/content/drive/MyDrive/dataset/naveen_images',
    '/content/drive/MyDrive/dataset/earring_best'
]

TARGET_SIZE = (224, 224)  # Resize images to this size
AUGMENTATIONS_PER_IMAGE = 5  # Number of augmented versions per original image

# Create output directories
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{BASE_OUTPUT_DIR}/images', exist_ok=True)

# Load the CSV file
print("Loading CSV file...")
df = pd.read_csv(CSV_FILE_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Sample data:")
print(df.head())

# Define augmentation parameters based on your requirements
datagen = ImageDataGenerator(
    rotation_range=90,           # 90° rotations
    width_shift_range=0.3,       # Image width shift by 30%
    height_shift_range=0.3,      # Image length shift by 30%
    zoom_range=[0.85, 1.05],     # Image enlargement/reduction by 15% (actually using zoom for "cuts")
    brightness_range=[0.2, 1.8], # Brightness range by 80% (0.2 = 80% darker, 1.8 = 80% brighter)
    horizontal_flip=True,        # Horizontal flips
    vertical_flip=True,          # Vertical flips
    fill_mode='nearest'          # How to fill in newly created pixels
)

# Additional augmentation function for color changes
def apply_color_shift(image_array, intensity=0.1):
    """Apply slight color changes to the image"""
    # Convert to float for processing
    img = image_array.astype(np.float32) / 255.0

    # Random color shifts for each channel
    color_shifts = np.random.uniform(-intensity, intensity, 3)

    for i in range(3):  # RGB channels
        img[:, :, i] = np.clip(img[:, :, i] + color_shifts[i], 0, 1)

    return (img * 255).astype(np.uint8)

# Function to find image in multiple folders
def find_image_path(image_filename, search_folders):
    """Find the full path of an image across multiple folders"""
    # If it's already a full path, return it
    if os.path.isabs(image_filename) and os.path.exists(image_filename):
        return image_filename

    # Extract just the filename if path includes folders
    base_filename = os.path.basename(image_filename)

    # Search in all folders
    for folder in search_folders:
        potential_path = os.path.join(folder, base_filename)
        if os.path.exists(potential_path):
            return potential_path

        # Also try the original filename in case it includes subfolder
        potential_path = os.path.join(folder, image_filename)
        if os.path.exists(potential_path):
            return potential_path

    return None

# Function to create a new filename for augmented image
def create_augmented_filename(original_path, aug_index, aug_type):
    """Create a new filename for augmented image"""
    base_name = os.path.splitext(os.path.basename(original_path))[0]
    extension = os.path.splitext(original_path)[1]
    return f"{base_name}_aug_{aug_index}_{aug_type}{extension}"


def process_single_image(img_path, output_dir, base_filename):
    """Process a single image and create augmented versions"""
    try:
        # Load and preprocess image
        if not os.path.exists(img_path):
            print(f"Warning: Image not found - {img_path}")
            return []

        # Load image
        img = load_img(img_path, target_size=TARGET_SIZE)
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)

        augmented_files = []

        # Generate augmented images
        aug_iter = datagen.flow(img_array, batch_size=1)

        for i in range(AUGMENTATIONS_PER_IMAGE):
            # Get augmented image
            aug_img = next(aug_iter)[0].astype(np.uint8)

            # Apply additional color shift (50% chance)
            if random.random() > 0.5:
                aug_img = apply_color_shift(aug_img, intensity=0.1)
                aug_type = f"color_shift_{i}"
            else:
                aug_type = f"geometric_{i}"

            # Create filename and save
            aug_filename = create_augmented_filename(base_filename, i, aug_type)
            aug_path = os.path.join(output_dir, aug_filename)

            # Save augmented image
            save_img(aug_path, aug_img)
            augmented_files.append(aug_filename)

        return augmented_files

    except Exception as e:
        print(f"Error processing {img_path}: {str(e)}")
        return []

# Create augmented dataset
print("Starting augmentation process...")
augmented_data = []
missing_images = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Augmenting images"):
    original_img_path = row['image_path']

    # Find the actual image path across multiple folders
    full_img_path = find_image_path(original_img_path, IMAGE_FOLDERS)

    if full_img_path is None:
        print(f"Warning: Image not found in any folder - {original_img_path}")
        missing_images.append(original_img_path)
        continue

    # Process the image
    base_filename = os.path.basename(original_img_path)
    augmented_files = process_single_image(full_img_path, f'{BASE_OUTPUT_DIR}/images', base_filename)

    # Add original image to new dataset
    augmented_data.append({
        'image_path': f'images/{base_filename}',
        'description': row['description'],
        'jewelry_type': row['jewelry_type'], # Corrected column name
        'is_augmented': False,
        'original_image': base_filename,
        'source_folder': os.path.dirname(full_img_path)
    })

    # Copy original image to output directory
    try:
        shutil.copy2(full_img_path, f'{BASE_OUTPUT_DIR}/images/{base_filename}')
    except Exception as e:
        print(f"Error copying original image {full_img_path}: {str(e)}")

    # Add augmented images to dataset
    for aug_file in augmented_files:
        augmented_data.append({
            'image_path': f'images/{aug_file}',
            'description': row['description'],  # Keep original description
            'jewelry_type': row['jewelry_type'],  # Keep original type (Corrected column name)
            'is_augmented': True,
            'original_image': base_filename,
            'source_folder': os.path.dirname(full_img_path)
        })

# Create new DataFrame with augmented data
augmented_df = pd.DataFrame(augmented_data)

# Save the new CSV
output_csv_path = f'{BASE_OUTPUT_DIR}/augmented_dataset.csv'
augmented_df.to_csv(output_csv_path, index=False)

print(f"\nAugmentation completed!")
print(f"Original dataset size: {len(df)} images")
print(f"Augmented dataset size: {len(augmented_df)} images")
print(f"Missing images: {len(missing_images)}")
if missing_images:
    print("Missing image files:")
    for missing in missing_images[:10]:  # Show first 10
        print(f"  - {missing}")
    if len(missing_images) > 10:
        print(f"  ... and {len(missing_images) - 10} more")

print(f"Augmentation factor: {len(augmented_df) / (len(df) - len(missing_images)):.2f}x")
print(f"\nFiles saved to: {BASE_OUTPUT_DIR}")
print(f"CSV file: {output_csv_path}")

# Display statistics
print(f"\nDataset statistics:")
print(f"Original images: {len(augmented_df[augmented_df['is_augmented'] == False])}")
print(f"Augmented images: {len(augmented_df[augmented_df['is_augmented'] == True])}")
print(f"\nType distribution:")
print(augmented_df['jewelry_type'].value_counts()) # Corrected column name
print(f"\nSource folder distribution:")
print(augmented_df[augmented_df['is_augmented'] == False]['source_folder'].value_counts())

# Function to visualize some augmentations (optional)
def visualize_augmentations(original_img_path, num_augmentations=4):
    """Visualize original image and its augmentations"""
    if not os.path.exists(original_img_path):
        print(f"Image not found: {original_img_path}")
        return

    # Load original image
    orig_img = load_img(original_img_path, target_size=TARGET_SIZE)
    orig_array = img_to_array(orig_img)
    orig_array = np.expand_dims(orig_array, axis=0)

    # Create figure
    fig, axes = plt.subplots(1, num_augmentations + 1, figsize=(20, 4))

    # Show original
    axes[0].imshow(orig_img)
    axes[0].set_title('Original')
    axes[0].axis('off')

    # Generate and show augmentations
    aug_iter = datagen.flow(orig_array, batch_size=1)

    for i in range(num_augmentations):
        aug_img = next(aug_iter)[0].astype(np.uint8)
        axes[i + 1].imshow(aug_img)
        axes[i + 1].set_title(f'Augmented {i + 1}')
        axes[i + 1].axis('off')

    plt.tight_layout()
    plt.show()

# Example usage of visualization (uncomment to use)
# if len(augmented_df) > 0:
#     # Find a sample image that exists
#     sample_row = augmented_df[augmented_df['is_augmented'] == False].iloc[0]
#     sample_img_path = find_image_path(sample_row['original_image'], IMAGE_FOLDERS)
#     if sample_img_path:
#         print(f"Visualizing augmentations for: {sample_img_path}")
#         visualize_augmentations(sample_img_path)

print("\nScript completed successfully!")
print(f"You can now use the augmented dataset from: {output_csv_path}")

In [ ]:
# Jewelry Dataset Augmentation for Google Colab
# Augments images based on CSV file with image_path, description, type columns

import pandas as pd
import numpy as np
import os
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, save_img
import matplotlib.pyplot as plt
from PIL import Image
import shutil
from tqdm import tqdm
import random

# Mount Google Drive (uncomment if your data is in Google Drive)
# from google.colab import drive
# drive.mount('/content/drive')

# Configuration
CSV_FILE_PATH = '/content/combined_jewelry_dataset - combined_jewelry_dataset.csv'  # Update this path
BASE_OUTPUT_DIR = '/content/Augmentation'  # Where to save augmented images

# Multiple image directories - Update these with your folder paths
IMAGE_FOLDERS = [
    '/content/drive/MyDrive/dataset/necklace_images',
    '/content/drive/MyDrive/dataset/necklace',
    '/content/drive/MyDrive/dataset/naveen_images',
    '/content/drive/MyDrive/dataset/earring_best'
]

TARGET_SIZE = (224, 224)  # Resize images to this size
AUGMENTATIONS_PER_IMAGE = 5  # Number of augmented versions per original image

# Create output directories
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{BASE_OUTPUT_DIR}/images', exist_ok=True)

# Load the CSV file
print("Loading CSV file...")
df = pd.read_csv(CSV_FILE_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Sample data:")
print(df.head())

# Define augmentation parameters
datagen = ImageDataGenerator(
    rotation_range=90,           # 90° rotations
    width_shift_range=0.3,       # Width shift 30%
    height_shift_range=0.3,      # Height shift 30%
    zoom_range=[0.85, 1.05],     # Enlargement/reduction by ~5-15%
    brightness_range=[0.2, 1.8], # Brightness range 80%
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

# Additional augmentation function for slight color changes
def apply_color_shift(image_array, intensity=0.1):
    img = image_array.astype(np.float32) / 255.0
    color_shifts = np.random.uniform(-intensity, intensity, 3)
    for i in range(3):
        img[:, :, i] = np.clip(img[:, :, i] + color_shifts[i], 0, 1)
    return (img * 255).astype(np.uint8)

# Find image in multiple folders
def find_image_path(image_filename, search_folders):
    if os.path.isabs(image_filename) and os.path.exists(image_filename):
        return image_filename
    base_filename = os.path.basename(image_filename)
    for folder in search_folders:
        potential_path = os.path.join(folder, base_filename)
        if os.path.exists(potential_path):
            return potential_path
        potential_path = os.path.join(folder, image_filename)
        if os.path.exists(potential_path):
            return potential_path
    return None

# Create augmented filename
def create_augmented_filename(original_path, aug_index, aug_type):
    base_name = os.path.splitext(os.path.basename(original_path))[0]
    extension = os.path.splitext(original_path)[1]
    return f"{base_name}_aug_{aug_index}_{aug_type}{extension}"

# Process single image
def process_single_image(img_path, output_dir, base_filename):
    try:
        if not os.path.exists(img_path):
            print(f"Warning: Image not found - {img_path}")
            return []
        img = load_img(img_path, target_size=TARGET_SIZE)
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)

        augmented_files = []
        aug_iter = datagen.flow(img_array, batch_size=1)

        for i in range(AUGMENTATIONS_PER_IMAGE):
            aug_img = next(aug_iter)[0].astype(np.uint8)
            if random.random() > 0.5:
                aug_img = apply_color_shift(aug_img, intensity=0.1)
                aug_type = f"color_shift_{i}"
            else:
                aug_type = f"geometric_{i}"
            aug_filename = create_augmented_filename(base_filename, i, aug_type)
            aug_path = os.path.join(output_dir, aug_filename)
            save_img(aug_path, aug_img)
            augmented_files.append(aug_filename)
        return augmented_files

    except Exception as e:
        print(f"Error processing {img_path}: {str(e)}")
        return []

# Start augmentation
print("Starting augmentation process...")
augmented_data = []
missing_images = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Augmenting images"):
    original_img_path = row['image_path']
    full_img_path = find_image_path(original_img_path, IMAGE_FOLDERS)

    if full_img_path is None:
        print(f"Warning: Image not found in any folder - {original_img_path}")
        missing_images.append(original_img_path)
        continue

    base_filename = os.path.basename(original_img_path)
    augmented_files = process_single_image(full_img_path, f'{BASE_OUTPUT_DIR}/images', base_filename)

    # Add original image
    augmented_data.append({
        'image_path': f'images/{base_filename}',
        'description': row['description'],
        'type': row['type'],  # ✅ fixed column
        'is_augmented': False,
        'original_image': base_filename,
        'source_folder': os.path.dirname(full_img_path)
    })

    # Copy original image
    try:
        shutil.copy2(full_img_path, f'{BASE_OUTPUT_DIR}/images/{base_filename}')
    except Exception as e:
        print(f"Error copying {full_img_path}: {str(e)}")

    # Add augmented images
    for aug_file in augmented_files:
        augmented_data.append({
            'image_path': f'images/{aug_file}',
            'description': row['description'],
            'type': row['type'],  # ✅ fixed column
            'is_augmented': True,
            'original_image': base_filename,
            'source_folder': os.path.dirname(full_img_path)
        })

# Save new dataset CSV
augmented_df = pd.DataFrame(augmented_data)
output_csv_path = f'{BASE_OUTPUT_DIR}/augmented_dataset.csv'
augmented_df.to_csv(output_csv_path, index=False)

print(f"\n✅ Augmentation completed!")
print(f"Original dataset size: {len(df)}")
print(f"Augmented dataset size: {len(augmented_df)}")
print(f"Missing images: {len(missing_images)}")

# Stats
print("\nType distribution:")
print(augmented_df['type'].value_counts())

print("\nAugmented images count:")
print(len(augmented_df[augmented_df['is_augmented'] == True]))
print("Original images count:")
print(len(augmented_df[augmented_df['is_augmented'] == False]))


In [ ]:
import pandas as pd
import numpy as np
import os
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, save_img
import matplotlib.pyplot as plt
from PIL import Image
import shutil
from tqdm import tqdm
import random

# Mount Google Drive if needed
# from google.colab import drive
# drive.mount('/content/drive')

# === Configuration ===
CSV_FILE_PATH = '/content/combined_jewelry_dataset - combined_jewelry_dataset.csv'  # Update this path
IMAGE_FOLDERS = [
    '/content/drive/MyDrive/dataset/necklace_images',
    '/content/drive/MyDrive/dataset/necklace',
    '/content/drive/MyDrive/dataset/naveen_images',
    '/content/drive/MyDrive/dataset/earring_best'
]
BASE_OUTPUT_DIR = '/content/Augmentation'  # Where to save augmented images
TARGET_SIZE = (224, 224)  # Resize images
AUGMENTATIONS_PER_IMAGE = 5  # Number of augmented versions per image

# === Prepare output directories ===
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{BASE_OUTPUT_DIR}/images', exist_ok=True)

# === Load CSV ===
print("Loading CSV file...")
df = pd.read_csv(CSV_FILE_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

# === Augmentation settings ===
datagen = ImageDataGenerator(
    rotation_range=90,
    width_shift_range=0.3,
    height_shift_range=0.3,
    zoom_range=[0.85, 1.05],
    brightness_range=[0.2, 1.8],
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

# === Extra color shift augmentation ===
def apply_color_shift(image_array, intensity=0.1):
    img = image_array.astype(np.float32) / 255.0
    color_shifts = np.random.uniform(-intensity, intensity, 3)
    for i in range(3):
        img[:, :, i] = np.clip(img[:, :, i] + color_shifts[i], 0, 1)
    return (img * 255).astype(np.uint8)

# === Recursive search for image ===
def find_image_path(image_filename, search_folders):
    base_filename = os.path.basename(image_filename).lower()
    for folder in search_folders:
        for root, _, files in os.walk(folder):
            for file in files:
                if file.lower() == base_filename:
                    return os.path.join(root, file)
    return None

# === Create new filename for augmented image ===
def create_augmented_filename(original_path, aug_index, aug_type):
    base_name = os.path.splitext(os.path.basename(original_path))[0]
    extension = os.path.splitext(original_path)[1]
    return f"{base_name}_aug_{aug_index}_{aug_type}{extension}"

# === Image augmentation process ===
def process_single_image(img_path, output_dir, base_filename):
    try:
        if not os.path.exists(img_path):
            print(f"Warning: Image not found - {img_path}")
            return []
        img = load_img(img_path, target_size=TARGET_SIZE)
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        augmented_files = []
        aug_iter = datagen.flow(img_array, batch_size=1)
        for i in range(AUGMENTATIONS_PER_IMAGE):
            aug_img = next(aug_iter)[0].astype(np.uint8)
            # Color shift augmentation (random)
            if random.random() > 0.5:
                aug_img = apply_color_shift(aug_img, intensity=0.1)
                aug_type = f"color_shift_{i}"
            else:
                aug_type = f"geometric_{i}"
            aug_filename = create_augmented_filename(base_filename, i, aug_type)
            aug_path = os.path.join(output_dir, aug_filename)
            save_img(aug_path, aug_img)
            augmented_files.append(aug_filename)
        return augmented_files
    except Exception as e:
        print(f"Error processing {img_path}: {str(e)}")
        return []

# === Augmentation main loop ===
print("Starting augmentation process...")
augmented_data = []
missing_images = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Augmenting images"):
    original_img_path = row['image_path']
    # Recursive search across all folders
    full_img_path = find_image_path(original_img_path, IMAGE_FOLDERS)
    if full_img_path is None:
        print(f"Warning: Image not found in any folder - {original_img_path}")
        missing_images.append(original_img_path)
        continue
    base_filename = os.path.basename(original_img_path)
    augmented_files = process_single_image(full_img_path, f'{BASE_OUTPUT_DIR}/images', base_filename)
    # Add original image record
    augmented_data.append({
        'image_path': f'images/{base_filename}',
        'description': row['description'],
        'jewelry_type': row['jewelry_type'],  # Ensure this matches your column name
        'is_augmented': False,
        'original_image': base_filename,
        'source_folder': os.path.dirname(full_img_path)
    })
    # Copy original file
    try:
        shutil.copy2(full_img_path, f'{BASE_OUTPUT_DIR}/images/{base_filename}')
    except Exception as e:
        print(f"Error copying original image {full_img_path}: {str(e)}")
    # Add augmented image records
    for aug_file in augmented_files:
        augmented_data.append({
            'image_path': f'images/{aug_file}',
            'description': row['description'],
            'jewelry_type': row['jewelry_type'],
            'is_augmented': True,
            'original_image': base_filename,
            'source_folder': os.path.dirname(full_img_path)
        })

# === Save results ===
augmented_df = pd.DataFrame(augmented_data)
output_csv_path = f'{BASE_OUTPUT_DIR}/augmented_dataset.csv'
augmented_df.to_csv(output_csv_path, index=False)
print(f"\nAugmentation completed!")
print(f"Original dataset size: {len(df)} images")
print(f"Augmented dataset size: {len(augmented_df)} images")
print(f"Missing images: {len(missing_images)}")
if missing_images:
    print("Missing image files:")
    for missing in missing_images[:10]:  # Show first 10
        print(f"  - {missing}")
    if len(missing_images) > 10:
        print(f"  ... and {len(missing_images) - 10} more")
print(f"Augmentation factor: {len(augmented_df) / (len(df) - len(missing_images)):.2f}x")
print(f"\nFiles saved to: {BASE_OUTPUT_DIR}")
print(f"CSV file: {output_csv_path}")

# === Dataset statistics ===
print(f"\nDataset statistics:")
print(f"Original images: {len(augmented_df[augmented_df['is_augmented'] == False])}")
print(f"Augmented images: {len(augmented_df[augmented_df['is_augmented'] == True])}")
print(f"\nType distribution:")
print(augmented_df['jewelry_type'].value_counts())
print(f"\nSource folder distribution:")
print(augmented_df[augmented_df['is_augmented'] == False]['source_folder'].value_counts())

# === Visualize augmentations for one image (optional) ===
def visualize_augmentations(original_img_path, num_augmentations=4):
    if not os.path.exists(original_img_path):
        print(f"Image not found: {original_img_path}")
        return
    orig_img = load_img(original_img_path, target_size=TARGET_SIZE)
    orig_array = img_to_array(orig_img)
    orig_array = np.expand_dims(orig_array, axis=0)
    fig, axes = plt.subplots(1, num_augmentations + 1, figsize=(20, 4))
    axes[0].imshow(orig_img)
    axes[0].set_title('Original')
    axes[0].axis('off')
    aug_iter = datagen.flow(orig_array, batch_size=1)
    for i in range(num_augmentations):
        aug_img = next(aug_iter)[0].astype(np.uint8)
        axes[i + 1].imshow(aug_img)
        axes[i + 1].set_title(f'Augmented {i + 1}')
        axes[i + 1].axis('off')
    plt.tight_layout()
    plt.show()

# Uncomment this to visualize a sample image and its augmentations
# if len(augmented_df) > 0:
#     sample_row = augmented_df[augmented_df['is_augmented'] == False].iloc[0]
#     sample_img_path = find_image_path(sample_row['original_image'], IMAGE_FOLDERS)
#     if sample_img_path:
#         print(f"Visualizing augmentations for: {sample_img_path}")
#         visualize_augmentations(sample_img_path)

print("\nScript completed successfully!")
print(f"You can now use the augmented dataset from: {output_csv_path}")


Loading CSV file...
Dataset shape: (5156, 3)
Columns: ['image_path', 'description', 'jewelry_type']
                 image_path  \
0  earring_best\001_001.png   
1  earring_best\001_002.png   
2  earring_best\001_003.png   
3  earring_best\001_004.png   
4  earring_best\001_005.png   

                                         description jewelry_type  
0  A pair of gold earrings with diamond accents i...      earring  
1  A pair of gold earrings with infinity-shaped p...      earring  
2  Elegant gold and diamond earrings with a leaf-...      earring  
3  Pair of gold earrings with diamond embellishme...      earring  
4  A pair of gold hoop earrings with diamond-stud...      earring  
Starting augmentation process...


Augmenting images:   0%|          | 2/5156 [00:00<04:43, 18.19it/s]

Augmenting images:   0%|          | 4/5156 [00:00<05:03, 16.99it/s]

Augmenting images:   0%|          | 6/5156 [00:00<05:06, 16.80it/s]

Augmenting images:   0%|          | 8/5156 [00:00<06:43, 12.77it/s]

Augmenting images:   0%|          | 10/5156 [00:00<08:34, 10.00it/s]

... (65 outputs left out)

Augmenting images:   5%|▌         | 268/5156 [00:18<05:04, 16.03it/s]

Augmenting images:   5%|▌         | 272/5156 [00:18<04:40, 17.39it/s]

Augmenting images:   5%|▌         | 276/5156 [00:19<04:28, 18.15it/s]

Augmenting images:   5%|▌         | 278/5156 [00:19<04:28, 18.18it/s]

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Jewelry Dataset Augmentation for Google Colab
# Augments images based on CSV file with image_path, description, jewelry_type columns

import pandas as pd
import numpy as np
import os
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, save_img
import matplotlib.pyplot as plt
from PIL import Image
import shutil
from tqdm import tqdm
import random

# Mount Google Drive if needed
# from google.colab import drive
# drive.mount('/content/drive')

# Configuration
CSV_FILE_PATH = '/content/combined_jewelry_dataset - combined_jewelry_dataset.csv'
BASE_OUTPUT_DIR = '/content/Augment'

IMAGE_FOLDERS = [
    '/content/drive/MyDrive/dataset/necklace_images',
    '/content/drive/MyDrive/dataset/necklace',
    '/content/drive/MyDrive/dataset/naveen_images',
    '/content/drive/MyDrive/dataset/earring_best',
    '/content/drive/MyDrive/dataset/earring_nature bg',

    '/content/drive/MyDrive/dataset/earring_person',
    '/content/drive/MyDrive/dataset/earringp'
]

TARGET_SIZE = (224, 224)
AUGMENTATIONS_PER_IMAGE = 5

# Create output directories
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{BASE_OUTPUT_DIR}/images', exist_ok=True)

# Load CSV
df = pd.read_csv(CSV_FILE_PATH)

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Fix backslashes in paths
df['image_path'] = df['image_path'].str.replace('\\', '/', regex=False)

print("Columns in CSV:", df.columns.tolist())
print("Sample data:\n", df.head())

# Define augmentation parameters
datagen = ImageDataGenerator(
    rotation_range=90,
    width_shift_range=0.3,
    height_shift_range=0.3,
    zoom_range=[0.85, 1.05],
    brightness_range=[0.2, 1.8],
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

def apply_color_shift(image_array, intensity=0.1):
    img = image_array.astype(np.float32) / 255.0
    color_shifts = np.random.uniform(-intensity, intensity, 3)
    for i in range(3):
        img[:, :, i] = np.clip(img[:, :, i] + color_shifts[i], 0, 1)
    return (img * 255).astype(np.uint8)

def find_image_path(image_filename, search_folders):
    if os.path.isabs(image_filename) and os.path.exists(image_filename):
        return image_filename
    base_filename = os.path.basename(image_filename)
    for folder in search_folders:
        potential_path = os.path.join(folder, base_filename)
        if os.path.exists(potential_path):
            return potential_path
        potential_path = os.path.join(folder, image_filename)
        if os.path.exists(potential_path):
            return potential_path
    return None

def create_augmented_filename(original_path, aug_index, aug_type):
    base_name = os.path.splitext(os.path.basename(original_path))[0]
    extension = os.path.splitext(original_path)[1]
    return f"{base_name}_aug_{aug_index}_{aug_type}{extension}"

def process_single_image(img_path, output_dir, base_filename):
    try:
        if not os.path.exists(img_path):
            print(f"Warning: Image not found - {img_path}")
            return []

        img = load_img(img_path, target_size=TARGET_SIZE)
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)

        augmented_files = []
        aug_iter = datagen.flow(img_array, batch_size=1)

        for i in range(AUGMENTATIONS_PER_IMAGE):
            aug_img = next(aug_iter)[0].astype(np.uint8)
            aug_type = f"color_shift_{i}" if random.random() > 0.5 else f"geometric_{i}"
            if "color_shift" in aug_type:
                aug_img = apply_color_shift(aug_img, intensity=0.1)
            aug_filename = create_augmented_filename(base_filename, i, aug_type)
            aug_path = os.path.join(output_dir, aug_filename)
            save_img(aug_path, aug_img)
            augmented_files.append(aug_filename)

        return augmented_files

    except Exception as e:
        print(f"Error processing {img_path}: {str(e)}")
        return []

# Augmentation loop
augmented_data = []
missing_images = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Augmenting images"):
    original_img_path = row['image_path']
    full_img_path = find_image_path(original_img_path, IMAGE_FOLDERS)

    if full_img_path is None:
        missing_images.append(original_img_path)
        print(f"Warning: Image not found in any folder - {original_img_path}")
        continue

    base_filename = os.path.basename(original_img_path)
    augmented_files = process_single_image(full_img_path, f'{BASE_OUTPUT_DIR}/images', base_filename)

    # Original image
    augmented_data.append({
        'image_path': f'images/{base_filename}',
        'description': row['description'],
        'jewelry_type': row['jewelry_type'],
        'is_augmented': False,
        'original_image': base_filename,
        'source_folder': os.path.dirname(full_img_path)
    })

    try:
        shutil.copy2(full_img_path, f'{BASE_OUTPUT_DIR}/images/{base_filename}')
    except Exception as e:
        print(f"Error copying original image {full_img_path}: {str(e)}")

    # Augmented images
    for aug_file in augmented_files:
        augmented_data.append({
            'image_path': f'images/{aug_file}',
            'description': row['description'],
            'jewelry_type': row['jewelry_type'],
            'is_augmented': True,
            'original_image': base_filename,
            'source_folder': os.path.dirname(full_img_path)
        })

# Save new CSV
augmented_df = pd.DataFrame(augmented_data)
output_csv_path = f'{BASE_OUTPUT_DIR}/augmented_dataset.csv'
augmented_df.to_csv(output_csv_path, index=False)

print(f"\nAugmentation completed! {len(augmented_df)} images saved.")
print(f"Missing images: {len(missing_images)}")
if missing_images:
    print("First 10 missing images:")
    print(missing_images[:10])
print(f"CSV file saved to: {output_csv_path}")

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import os
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, save_img
import matplotlib.pyplot as plt
import shutil
from tqdm import tqdm
import random

# Configurations
CSV_FILE_PATH = '/content/combined_jewelry_dataset - combined_jewelry_dataset.csv'  # Path to your CSV
IMAGE_FOLDERS = [
    '/content/drive/MyDrive/dataset/necklace_images',
    '/content/drive/MyDrive/dataset/necklace',
    '/content/drive/MyDrive/dataset/naveen_images',
    '/content/drive/MyDrive/dataset/earring_best'
]
BASE_OUTPUT_DIR = '/content/Augmentation'
TARGET_SIZE = (224, 224)
AUGMENTATIONS_PER_IMAGE = 5
COMMON_EXTENSIONS = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.JPG', '.JPEG', '.PNG']

os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{BASE_OUTPUT_DIR}/images', exist_ok=True)

print("Loading CSV file...")
df = pd.read_csv(CSV_FILE_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

datagen = ImageDataGenerator(
    rotation_range=90,
    width_shift_range=0.3,
    height_shift_range=0.3,
    zoom_range=[0.85, 1.05],
    brightness_range=[0.2, 1.8],
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

def apply_color_shift(image_array, intensity=0.1):
    img = image_array.astype(np.float32) / 255.0
    color_shifts = np.random.uniform(-intensity, intensity, 3)
    for i in range(3):
        img[:, :, i] = np.clip(img[:, :, i] + color_shifts[i], 0, 1)
    return (img * 255).astype(np.uint8)

def find_image_path(image_filename, search_folders, extensions=COMMON_EXTENSIONS):
    base_filename = os.path.splitext(os.path.basename(str(image_filename)).strip())[0].lower()
    for folder in search_folders:
        for root, _, files in os.walk(folder):
            for file in files:
                file_lower = file.lower()
                if file_lower == base_filename or os.path.splitext(file_lower)[0] == base_filename:
                    return os.path.join(root, file)
                # Flexible extension match
                if os.path.splitext(file_lower)[0] == base_filename and os.path.splitext(file_lower)[1] in extensions:
                    return os.path.join(root, file)
            # Try with various extensions if not found with given
            for ext in extensions:
                candidate = os.path.join(root, base_filename + ext)
                if os.path.exists(candidate):
                    return candidate
    return None

def create_augmented_filename(original_path, aug_index, aug_type):
    base_name = os.path.splitext(os.path.basename(original_path))[0]
    extension = os.path.splitext(original_path)[1]
    if not extension:
        extension = ".jpg"
    return f"{base_name}_aug_{aug_index}_{aug_type}{extension}"

def process_single_image(img_path, output_dir, base_filename):
    try:
        if not os.path.exists(img_path):
            print(f"Warning: Image not found - {img_path}")
            return []
        img = load_img(img_path, target_size=TARGET_SIZE)
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        augmented_files = []
        aug_iter = datagen.flow(img_array, batch_size=1)
        for i in range(AUGMENTATIONS_PER_IMAGE):
            aug_img = next(aug_iter)[0].astype(np.uint8)
            if random.random() > 0.5:
                aug_img = apply_color_shift(aug_img, intensity=0.1)
                aug_type = f"color_shift_{i}"
            else:
                aug_type = f"geometric_{i}"
            aug_filename = create_augmented_filename(base_filename, i, aug_type)
            aug_path = os.path.join(output_dir, aug_filename)
            save_img(aug_path, aug_img)
            augmented_files.append(aug_filename)
        return augmented_files
    except Exception as e:
        print(f"Error processing {img_path}: {str(e)}")
        return []

print("Starting augmentation process...")
augmented_data = []
missing_images = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Augmenting images"):
    original_img_path = str(row['image_path']).strip()  # Remove stray spaces
    full_img_path = find_image_path(original_img_path, IMAGE_FOLDERS)
    if full_img_path is None:
        print(f"Missing: {original_img_path}")
        missing_images.append(original_img_path)
        continue
    base_filename = os.path.basename(original_img_path)
    augmented_files = process_single_image(full_img_path, f'{BASE_OUTPUT_DIR}/images', base_filename)
    augmented_data.append({
        'image_path': f'images/{base_filename}',
        'description': row['description'],
        'jewelry_type': row['jewelry_type'],
        'is_augmented': False,
        'original_image': base_filename,
        'source_folder': os.path.dirname(full_img_path)
    })
    try:
        shutil.copy2(full_img_path, f'{BASE_OUTPUT_DIR}/images/{base_filename}')
    except Exception as e:
        print(f"Error copying original image {full_img_path}: {str(e)}")
    for aug_file in augmented_files:
        augmented_data.append({
            'image_path': f'images/{aug_file}',
            'description': row['description'],
            'jewelry_type': row['jewelry_type'],
            'is_augmented': True,
            'original_image': base_filename,
            'source_folder': os.path.dirname(full_img_path)
        })

augmented_df = pd.DataFrame(augmented_data)
output_csv_path = f'{BASE_OUTPUT_DIR}/augmented_dataset.csv'
augmented_df.to_csv(output_csv_path, index=False)
print("\nAugmentation completed!")
print(f"Original dataset size: {len(df)} images")
print(f"Augmented dataset size: {len(augmented_df)} images")
print(f"Missing images: {len(missing_images)}")
if missing_images:
    print("Missing image files:")
    for missing in missing_images[:10]:
        print(f"  - {missing}")
    if len(missing_images) > 10:
        print(f"  ... and {len(missing_images) - 10} more")
print(f"Augmentation factor: {len(augmented_df) / (len(df) - len(missing_images)):.2f}x")
print(f"\nFiles saved to: {BASE_OUTPUT_DIR}")
print(f"CSV file: {output_csv_path}")

print("\nDataset statistics:")
print(f"Original images: {len(augmented_df[augmented_df['is_augmented']==False])}")
print(f"Augmented images: {len(augmented_df[augmented_df['is_augmented']==True])}")
print("\nType distribution:")
print(augmented_df['jewelry_type'].value_counts())
print("\nSource folder distribution:")
print(augmented_df[augmented_df['is_augmented']==False]['source_folder'].value_counts())

print("\nScript completed successfully!")
print(f"You can now use the augmented dataset from: {output_csv_path}")


Loading CSV file...
Dataset shape: (5156, 3)
Columns: ['image_path', 'description', 'jewelry_type']
                 image_path  \
0  earring_best\001_001.png   
1  earring_best\001_002.png   
2  earring_best\001_003.png   
3  earring_best\001_004.png   
4  earring_best\001_005.png   

                                         description jewelry_type  
0  A pair of gold earrings with diamond accents i...      earring  
1  A pair of gold earrings with infinity-shaped p...      earring  
2  Elegant gold and diamond earrings with a leaf-...      earring  
3  Pair of gold earrings with diamond embellishme...      earring  
4  A pair of gold hoop earrings with diamond-stud...      earring  
Starting augmentation process...


Augmenting images:   0%|          | 2/5156 [00:00<24:16,  3.54it/s]

Missing: earring_best\001_001.png
Missing: earring_best\001_002.png


Augmenting images:   0%|          | 4/5156 [00:00<15:11,  5.65it/s]

Missing: earring_best\001_003.png
Missing: earring_best\001_004.png


Augmenting images:   0%|          | 6/5156 [00:01<11:53,  7.22it/s]

... (123 outputs left out)

Augmenting images:   2%|▏         | 116/5156 [00:21<38:52,  2.16it/s]

Missing: earring_best\003_049.png
Missing: earring_best\003_050.png


Augmenting images:   2%|▏         | 117/5156 [00:22<15:49,  5.31it/s]


KeyboardInterrupt: 

In [4]:
import pandas as pd
import numpy as np
import os
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, save_img
import matplotlib.pyplot as plt
import shutil
from tqdm import tqdm
import random

# Configurations
CSV_FILE_PATH = '/content/combined_jewelry_dataset - combined_jewelry_dataset (2).csv'  # Path to your CSV
IMAGE_FOLDERS = [
    '/content/drive/MyDrive/dataset/necklace_images',
    '/content/drive/MyDrive/dataset/necklace',
    '/content/drive/MyDrive/dataset/naveen_images',
    '/content/drive/MyDrive/dataset/earring_best',
    '/content/drive/MyDrive/dataset/earring_nature bg',
    '/content/drive/MyDrive/dataset/earring_person',
    '/content/drive/MyDrive/dataset/earringp',
    '/content/drive/MyDrive/dataset/earring_jhumkas',
    '/content/drive/MyDrive/dataset/earrings_randomish'
]
BASE_OUTPUT_DIR = '/content/Augment'
TARGET_SIZE = (224, 224)
AUGMENTATIONS_PER_IMAGE = 5
COMMON_EXTENSIONS = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.JPG', '.JPEG', '.PNG']

os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{BASE_OUTPUT_DIR}/images', exist_ok=True)

print("Loading CSV file...")
df = pd.read_csv(CSV_FILE_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

datagen = ImageDataGenerator(
    rotation_range=90,
    width_shift_range=0.3,
    height_shift_range=0.3,
    zoom_range=[0.85, 1.05],
    brightness_range=[0.2, 1.8],
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

def apply_color_shift(image_array, intensity=0.1):
    img = image_array.astype(np.float32) / 255.0
    color_shifts = np.random.uniform(-intensity, intensity, 3)
    for i in range(3):
        img[:, :, i] = np.clip(img[:, :, i] + color_shifts[i], 0, 1)
    return (img * 255).astype(np.uint8)

def find_image_path(image_filename, search_folders, extensions=COMMON_EXTENSIONS):
    target_base = os.path.splitext(os.path.basename(image_filename))[0].lower()

    for folder in search_folders:
        for root, _, files in os.walk(folder):
            for file in files:
                normalized_file = os.path.splitext(file)[0].lower()
                if normalized_file == target_base:
                    return os.path.join(root, file)
            # Extension fallback search
            for ext in extensions:
                candidate = os.path.join(root, target_base + ext)
                if os.path.exists(candidate):
                    return candidate
    return None

def create_augmented_filename(original_path, aug_index, aug_type):
    base_name = os.path.splitext(os.path.basename(original_path))[0]
    extension = os.path.splitext(original_path)[1]
    if not extension:
        extension = ".jpg"
    return f"{base_name}_aug_{aug_index}_{aug_type}{extension}"

def process_single_image(img_path, output_dir, base_filename):
    try:
        if not os.path.exists(img_path):
            print(f"Warning: Image not found - {img_path}")
            return []
        img = load_img(img_path, target_size=TARGET_SIZE)
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        augmented_files = []
        aug_iter = datagen.flow(img_array, batch_size=1)
        for i in range(AUGMENTATIONS_PER_IMAGE):
            aug_img = next(aug_iter)[0].astype(np.uint8)
            if random.random() > 0.5:
                aug_img = apply_color_shift(aug_img, intensity=0.1)
                aug_type = f"color_shift_{i}"
            else:
                aug_type = f"geometric_{i}"
            aug_filename = create_augmented_filename(base_filename, i, aug_type)
            aug_path = os.path.join(output_dir, aug_filename)
            save_img(aug_path, aug_img)
            augmented_files.append(aug_filename)
        return augmented_files
    except Exception as e:
        print(f"Error processing {img_path}: {str(e)}")
        return []

print("Starting augmentation process...")
augmented_data = []
missing_images = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Augmenting images"):
    # Normalize path with replacement and os.path.normpath
    raw_path = str(row['image_path']).strip()
    normalized_path = os.path.normpath(raw_path.replace('\\', '/'))

    full_img_path = find_image_path(normalized_path, IMAGE_FOLDERS)
    if full_img_path is None:
        print(f"Missing: {normalized_path}")
        missing_images.append(normalized_path)
        continue

    base_filename = os.path.basename(normalized_path)
    augmented_files = process_single_image(full_img_path, f'{BASE_OUTPUT_DIR}/images', base_filename)

    augmented_data.append({
        'image_path': f'images/{base_filename}',
        'description': row['description'],
        'jewelry_type': row['jewelry_type'],
        'is_augmented': False,
        'original_image': base_filename,
        'source_folder': os.path.dirname(full_img_path)
    })

    try:
        shutil.copy2(full_img_path, f'{BASE_OUTPUT_DIR}/images/{base_filename}')
    except Exception as e:
        print(f"Error copying original image {full_img_path}: {str(e)}")

    for aug_file in augmented_files:
        augmented_data.append({
            'image_path': f'images/{aug_file}',
            'description': row['description'],
            'jewelry_type': row['jewelry_type'],
            'is_augmented': True,
            'original_image': base_filename,
            'source_folder': os.path.dirname(full_img_path)
        })

augmented_df = pd.DataFrame(augmented_data)
output_csv_path = f'{BASE_OUTPUT_DIR}/augmented_dataset.csv'
augmented_df.to_csv(output_csv_path, index=False)

print("\nAugmentation completed!")
print(f"Original dataset size: {len(df)} images")
print(f"Augmented dataset size: {len(augmented_df)} images")
print(f"Missing images: {len(missing_images)}")
if missing_images:
    print("Missing image files:")
    for missing in missing_images[:10]:
        print(f"  - {missing}")
    if len(missing_images) > 10:
        print(f"  ... and {len(missing_images) - 10} more")
print(f"Augmentation factor: {len(augmented_df) / (len(df) - len(missing_images)):.2f}x")
print(f"\nFiles saved to: {BASE_OUTPUT_DIR}")
print(f"CSV file: {output_csv_path}")

print("\nDataset statistics:")
print(f"Original images: {len(augmented_df[augmented_df['is_augmented'] == False])}")
print(f"Augmented images: {len(augmented_df[augmented_df['is_augmented'] == True])}")
print("\nType distribution:")
print(augmented_df['jewelry_type'].value_counts())
print("\nSource folder distribution:")
print(augmented_df[augmented_df['is_augmented'] == False]['source_folder'].value_counts())

print("\nScript completed successfully!")
print(f"You can now use the augmented dataset from: {output_csv_path}")


Loading CSV file...
Dataset shape: (5407, 3)
Columns: ['image_path', 'description', 'jewelry_type']
                 image_path  \
0  earring_best/001_001.png   
1  earring_best/001_002.png   
2  earring_best/001_003.png   
3  earring_best/001_004.png   
4  earring_best/001_005.png   

                                         description jewelry_type  
0  A pair of gold earrings with diamond accents i...      earring  
1  A pair of gold earrings with infinity-shaped p...      earring  
2  Elegant gold and diamond earrings with a leaf-...      earring  
3  Pair of gold earrings with diamond embellishme...      earring  
4  A pair of gold hoop earrings with diamond-stud...      earring  
Starting augmentation process...


Augmenting images:   5%|▌         | 274/5407 [03:18<1:07:15,  1.27it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Augmenting images: 100%|██████████| 5407/5407 [28:36<00:00,  3.15it/s]



Augmentation completed!
Original dataset size: 5407 images
Augmented dataset size: 32442 images
Missing images: 0
Augmentation factor: 6.00x

Files saved to: /content/Augment
CSV file: /content/Augment/augmented_dataset.csv

Dataset statistics:
Original images: 5407
Augmented images: 27035

Type distribution:
jewelry_type
earring     21294
necklace    11148
Name: count, dtype: int64

Source folder distribution:
source_folder
/content/drive/MyDrive/dataset/earring_best          3335
/content/drive/MyDrive/dataset/necklace              1737
/content/drive/MyDrive/dataset/necklace_images        273
/content/drive/MyDrive/dataset/earring_jhumkas         61
/content/drive/MyDrive/dataset/earrings_randomish       1
Name: count, dtype: int64

Script completed successfully!
You can now use the augmented dataset from: /content/Augment/augmented_dataset.csv


In [5]:
!zip -r /content/augment_images.zip /content/Augment/images


Streaming output truncated to the last 5000 lines.
  adding: content/Augment/images/necklace_289_aug_1_geometric_1.jpg (deflated 4%)
  adding: content/Augment/images/product-hugerect-449558-169254-1423069448-3db6c3d4b4887a223a91e64c58b6d3e5_aug_0_geometric_0.jpg (deflated 19%)
  adding: content/Augment/images/010_030_aug_1_color_shift_1.png (deflated 1%)
  adding: content/Augment/images/necklace_75_aug_0_geometric_0.jpg (deflated 21%)
  adding: content/Augment/images/009_016_aug_2_geometric_2.png (stored 0%)
  adding: content/Augment/images/021_075_aug_3_geometric_3.png (deflated 1%)
  adding: content/Augment/images/HGVQTRHGJIO7_aug_0_color_shift_0.jpg (deflated 4%)
  adding: content/Augment/images/images (77)_aug_4_geometric_4.jpeg (deflated 5%)
  adding: content/Augment/images/007_011_aug_0_color_shift_0.png (deflated 2%)
  adding: content/Augment/images/LA0JTQARRFYX_aug_2_color_shift_2.jpg (deflated 4%)
  adding: content/Augment/images/008_019_aug_2_color_shift_2.png (deflated 2%)
 

In [7]:
from google.colab import files
files.download('/content/augment_images.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>